# Processamento dos slides

Primeiramente, a conversão é de PPTX para TXT. É possível pular essa etapa e ir diretamente para JSON, mas a conversão para TXT é útil para visualização e depuração.

In [ ]:
from glob import glob
from tqdm import tqdm
from pathlib import Path
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
import unicodedata

files = glob(str(Path("slides_adapt") / "*.pptx"))
files

In [ ]:
file = files[-1]
prs = Presentation(file)
prs

In [ ]:
print(f"Slides count: {len(prs.slides)}")
print(f"Slide width: {prs.slide_width}")
print(f"Slide height: {prs.slide_height}")
print(f"\nCore properties:")
print(f"  Title: {prs.core_properties.title}")
print(f"  Author: {prs.core_properties.author}")
print(f"  Subject: {prs.core_properties.subject}")
print(f"  Created: {prs.core_properties.created}")
print(f"  Modified: {prs.core_properties.modified}")

# Detailed slide information
print("\n=== Detailed Slide Information ===\n")
for slide_idx, slide in enumerate(prs.slides):
    print(f"\n--- Slide {slide_idx} ---")
    print(f"Shapes count: {len(slide.shapes)}")
    
    for shape_idx, shape in enumerate(slide.shapes):
        print(f"\n  Shape {shape_idx}:")
        print(f"    Type: {shape.shape_type} ({shape.shape_type.name if hasattr(shape.shape_type, 'name') else 'N/A'})")
        print(f"    Name: {shape.name}")
        print(f"    Left: {shape.left}, Top: {shape.top}")
        print(f"    Width: {shape.width}, Height: {shape.height}")
        
        if hasattr(shape, 'text') and shape.text:
            print(f"    Text: {repr(shape.text[:100])}{'...' if len(shape.text) > 100 else ''}")
        
        # if hasattr(shape, 'auto_shape_type'):
        if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
            print(f"    Auto Shape Type: {shape.auto_shape_type}")


In [ ]:
for shape in prs.slides[3].shapes:
    print(f"SHAPE_{shape.shape_type}")

    if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
        print(f"_AUTO_SHAPE_{shape.auto_shape_type}\n")
        print(f"HEIGHT_{shape.height}\n")
        print(f"TOP_{shape.top}\n")

    if hasattr(shape, "text"):
        print(f"\nSTART_TEXT\n{shape.text}\nEND_TEXT\n")
        print(f"HEIGHT_{shape.height}\n")
        print(f"TOP_{shape.top}\n")

    print("\n")

In [ ]:
def normalize_text(text):
    return "".join(
        ch
        for ch in unicodedata.normalize("NFD", text)
        if unicodedata.category(ch) != "Mn"
    )


for file in files:
    new_file = Path("slides_txt") / (Path(file).stem + ".txt")

    f = open(new_file, "w", encoding="utf-8")

    prs = Presentation(file)
    for i, slide in tqdm(enumerate(prs.slides), desc="Processing slides", unit="slide"):

        f.write(f"\nSLIDE_{i}\n")

        for shape in slide.shapes:
            f.write(f"SHAPE_{shape.shape_type}")

            if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
                f.write(f"_AUTO_SHAPE_{shape.auto_shape_type}\n")
                f.write(f"HEIGHT_{shape.height}\n")
                f.write(f"TOP_{shape.top}\n")

            if hasattr(shape, "text"):
                f.write(f"\nSTART_TEXT\n{shape.text}\nEND_TEXT\n")
                f.write(f"HEIGHT_{shape.height}\n")
                f.write(f"TOP_{shape.top}\n")

            f.write("\n")

        for shape in slide.shapes:
            if hasattr(shape, "text"):
                if normalize_text(shape.text).strip().lower() == "indice":
                    f.write("\n__END__\n")

    f.close()

# Processamento do texto

In [ ]:
import re
import json

files_txt = glob(str(Path("slides_txt") / "*.txt"))
print(f"Files found: {files_txt}")

In [ ]:
with open(files_txt[-1], "r", encoding="utf-8") as f:
    text = f.read()

print(text.split("__END__")[0])

In [ ]:
file_txt = files_txt[-1]
auto_shape_pattern = r"_AUTO_SHAPE_([A-Z_]+)"


print(f"Processing file: {file_txt}")

with open(file_txt, "r", encoding="utf-8") as f:
    text = f.read()

praise = text.split("__END__")[0]
praise_struc = {}
praise_struc["slides"] = []

for slide in praise.split("SLIDE_"):
    slide_struc = {}
    slide_num = slide.split("\n")[0].strip()
    if not slide_num:
        continue

    slide_struc["slide"] = slide_num
    slide_struc["shapes"] = []

    for shape in slide.split("\nSHAPE_"):
        record_line = False
        text_inside = ""
        shapes_struc = {}

        for line in shape.split("\n"):
            if "SHAPE_" in line:
                match = re.search(auto_shape_pattern, line)
                auto_shape = match.group(1)

                shapes_struc["auto_shape"] = auto_shape
                shapes_struc["shape"] = "AUTO_SHAPE"

            if "HEIGHT_" in line:
                height = line.split("_")[1]
                shapes_struc["height"] = int(height)
            if "TOP_" in line:
                top = line.split("_")[1]
                shapes_struc["top"] = int(top)

            if "END_TEXT" in line:
                record_line = False

            if record_line:
                text_inside += line + "\n"
            else:
                if text_inside:
                    shapes_struc["text"] = text_inside.strip()

            if "START_TEXT" in line:
                text_inside = ""
                record_line = True

        if shapes_struc:
            slide_struc["shapes"].append(shapes_struc)

    praise_struc["slides"].append(slide_struc)

print(json.dumps(praise_struc, ensure_ascii=False, indent=4))

# Processamento do JSON

In [ ]:
def get_texts(d):
    if isinstance(d, dict):
        for k, v in d.items():
            if k == "text":
                yield v
            else:
                yield from get_texts(v)
    elif isinstance(d, list):
        for v in d:
            yield from get_texts(v)

In [ ]:
import json

with open("slides_json\\LOUVORES AVULSOS_Rev_31.12.22_ADAPT.pptx.txt.json", "r") as f:
    praises = json.load(f)

praises

In [ ]:
erros_conhecidos = [
    "ABENÇOA-NOS SENHOR, DERRAMA SOBRE NÓS TUA PAZ ABENÇOA-NOS SENHOR, DERRAMA SOBRE NÓS TEU AMOR.",  # indice extra
    "MEU JESUS, SALVADOR,  OUTRO IGUAL NÃO HÁ. TODOS OS DIAS QUERO LOUVAR  AS MARAVILHAS DE TEU AMOR. CONSOLO, ABRIGO,  FORÇA E REFÚGIO É O SENHOR. COM TODO O MEU SER,  COM TUDO O QUE SOU,  SEMPRE TE ADORAREI.",  # pedaco de louvor
    "PODES CLAMAR, PODES CHORAR, EU ESTAREI PRONTO PRA TE AJUDAR, E SE O CORAÇÃO DESFALECER, CONFIA EM MIM SOU JESUS  E TE FAÇO VENCER.",  # pedaco de louvor
    "AO CORDEIRO GLÓRIA E HONRA,  SALVOS NÃO CESSEIS DE DAR. GLÓRIA, HONRA SEMPRE A DEUS ENTOAREI!  AMÉM!",  # indice extra
]
TAGS_LITERAIS = [
    "TODOS",
    "M",
    "H",
    "T",
    "BIS",
    "VARÕES",
    "SERVAS",
]
TAGS_CONTROLE = [
    "ÍNDICE",
    "CORO (2X)",
    "\n\nCORO",
    "CORO\n",
    "1X",
    "2X",
    "3X",
    "4X",
    "()",
    "(TODOS)",
    "(M)",
    "(H)",
    "(T)",
    "(BIS)",
    "(VARÕES)",
    "(SERVAS)",
    "REPETIR O LOUVOR",
    "REPETIR 1ª ESTROFE",
    "FINAL:",
    "BIS NO FINAL",
    "IGREJA CRISTÃ MARANATA",
    "ATUALIZAÇÃO",
    "\nINSTRUMENTOS",
]

In [ ]:
import re


def return_possible_title(texts):
    possible_title = [
        text
        for text in texts
        if all(opt not in text.upper() for opt in TAGS_CONTROLE)
        and text.strip() != ""
        and text.upper() not in TAGS_LITERAIS
    ]
    if possible_title:
        min_string = min(possible_title, key=len)
        title_index = texts.index(min_string)
        title = min_string.upper().replace("\n", " ")
        if title not in erros_conhecidos:
            return title, title_index
    return None, None


def set_text_clean(texts_wo_title):
    texts_clean = [re.sub(r"\s+", " ", text) for text in texts_wo_title]
    texts_clean = [line for line in texts_clean if line not in TAGS_LITERAIS]
    new_texts_clean = []
    for line in texts_clean:
        for tag in TAGS_CONTROLE:
            line = line.upper().replace(tag, "")
        line = line.strip()
        if line:
            new_texts_clean.append(line)
    texts_clean = new_texts_clean
    # texts_clean = list(dict.fromkeys(texts_clean))
    texts_clean = " ".join(texts_clean)
    texts_clean = texts_clean.replace("\n", " ")
    # remove all double quotes
    texts_clean = texts_clean.replace("“", "")
    texts_clean = texts_clean.replace("”", "")
    texts_clean = texts_clean.replace('"', "")
    return texts_clean


def set_text_full(texts_wo_title):
    texts_full = [
        line
        for line in texts_wo_title
        if line.upper() != "ÍNDICE" and line.strip() != ""
    ]
    texts_full = "\n\n".join(texts_full)
    texts_full = texts_full.replace("\n\nBIS", "\nBIS")
    texts_full = texts_full.replace('"', "'")
    return texts_full


def return_number(title):
    if title:
        for word in title.split(" "):
            match = re.search(r"\d+", word)
            if match:
                return match.group()
    return "null"


def process_structure(praises):
    new_structure = []
    for praise in praises:
        texts = list(get_texts(praise))

        if not texts:
            continue

        # clean double or more spaces in string array
        texts = [text.replace("–", "-") for text in texts]

        title, title_index = return_possible_title(texts)
        numero = return_number(title)

        texts_wo_title = texts.copy()
        if title_index is not None:
            texts_wo_title.pop(title_index)

        texts_full = set_text_full(texts_wo_title)
        texts_clean = set_text_clean(texts_wo_title)

        new_structure.append(
            {
                "numero": numero,
                "nome": title if title is not None else "null",
                "texto": texts_full,
                "texto_limpo": texts_clean,
            }
        )

    return new_structure


new_structure = process_structure(praises)

new_structure

# Criar arquivo de inserção

In [ ]:
import glob

files_json = glob.glob("slides_json\\*.json")
files_json

In [ ]:
for index, file in enumerate(files_json):
    file_name = (
        "00"
        + str(index + 3)
        + "-"
        + file.split("\\")[1].split("pptx")[0].lower().replace(" ", "_")
        + "sql"
    )

    with open(file, "r") as f:
        louvores = json.load(f)

    louvores_estruturados = process_structure(louvores)
    with open("db\\migrations\\" + file_name, "w") as f:
        for hino in louvores_estruturados:
            text = (
                "INSERT INTO hino (numero, nome, texto, texto_limpo, coletanea_id, date_insert, date_update) VALUES ('"
                + hino["numero"]
                + "', '"
                + hino["nome"]
                + "', '"
                + hino["texto"].replace("\n", "\\n").replace("'", "''")
                + "', '"
                + hino["texto_limpo"].replace("'", "''")
                + "', "
                + str(index + 1)
                + ", CURRENT_TIMESTAMP, CURRENT_TIMESTAMP);\n"
            )
            f.write(text)